# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
missing=[p for p in ['onnx','onnxruntime','onnxscript','torch','numpy'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('dependencies ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 9.0 MB/s eta 0:00:00
dependencies ok


In [4]:
import json, os, time, hashlib, zipfile,  csv, base64, pickle
import glob, sys,math, random, collections,io,shutil
from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort
import torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict, Counter
from onnx import shape_inference

In [5]:
torch.set_num_threads(1)

In [6]:
TASK_ID = "task382"
CH = 10
H = W = 30
LOCAL_TASK_JSON = Path("/mnt/data/task382.json")
KAGGLE_TASK_JSON = Path(COMPETITION) / "task382.json"
TASK_JSON = LOCAL_TASK_JSON if LOCAL_TASK_JSON.exists() else KAGGLE_TASK_JSON
OUT_DIR = Path.cwd() / "task382_triangular_wave_stripe_active_canvas_30x30"
OUT_DIR.mkdir(parents=True, exist_ok=True)
ONNX_PATH = OUT_DIR / f"{TASK_ID}.onnx"
SUBMISSION_PATH = Path.cwd() / "submission.zip"
SUMMARY_PATH = OUT_DIR / f"{TASK_ID}_triangular_wave_stripe_active_canvas_30x30_validation_summary.json"
RULE_DESCRIPTION = 'bouncing diagonal / triangular-wave stripe propagation + active-canvas masking'

with TASK_JSON.open("r") as f:
    task = json.load(f)

len(task["train"]), len(task["test"]), len(task["arc-gen"])


(3, 1, 262)

In [7]:

def grid_to_tensor_zero_padded(grid, h=H, w=W, ch=CH):
    """Convert an ARC grid to [1,10,30,30].
    Inside the real grid: one-hot, including background color 0.
    Outside the real grid: all-zero across every channel.
    """
    x = np.zeros((1, ch, h, w), dtype=np.float32)
    for r, row in enumerate(grid):
        for c, v in enumerate(row):
            x[0, int(v), r, c] = 1.0
    return x


def _infer_sides(arr):
    h, w = arr.shape
    pts8 = np.argwhere(arr == 8)
    pts2 = np.argwhere(arr == 2)

    def sides_for(pts):
        sides = []
        if len(pts) == 0:
            return sides
        if np.all(pts[:, 0] == 0):
            sides.append("top")
        if np.all(pts[:, 0] == h - 1):
            sides.append("bottom")
        if np.all(pts[:, 1] == 0):
            sides.append("left")
        if np.all(pts[:, 1] == w - 1):
            sides.append("right")
        return sides

    anchor_sides = sides_for(pts8)
    marker_sides = sides_for(pts2)

    for a in ["top", "bottom"]:
        for m in ["left", "right"]:
            if a in anchor_sides and m in marker_sides:
                return a, m
    for a in ["left", "right"]:
        for m in ["top", "bottom"]:
            if a in anchor_sides and m in marker_sides:
                return a, m
    raise ValueError("Could not infer compatible border sides")

def python_rule(grid):
    """Border-anchor triangular-wave stripe propagation.

    Color 8 anchors define the base stripe on one border. Color 2 markers on a
    perpendicular border shift the stripe by one cell each time the propagation
    crosses a marker. Marker pixels are preserved and padding is not considered
    part of the canvas.
    """
    arr = np.array(grid, dtype=np.int64)
    h, w = arr.shape
    out = np.zeros_like(arr)
    pts8 = np.argwhere(arr == 8)
    pts2 = np.argwhere(arr == 2)
    anchor_side, marker_side = _infer_sides(arr)

    if anchor_side in ["top", "bottom"]:
        base_cols = sorted(set(int(c) for r, c in pts8 if (r == 0 if anchor_side == "top" else r == h - 1)))
        marker_rows = sorted(set(int(r) for r, c in pts2 if (c == 0 if marker_side == "left" else c == w - 1)))
        sign = 1 if marker_side == "left" else -1
        for r in range(h):
            crossed = sum(1 for mr in marker_rows if (mr <= r if anchor_side == "top" else mr >= r))
            shift = sign * crossed
            for bc in base_cols:
                c = bc + shift
                if 0 <= c < w and arr[r, c] != 2:
                    out[r, c] = 8
    else:
        base_rows = sorted(set(int(r) for r, c in pts8 if (c == 0 if anchor_side == "left" else c == w - 1)))
        marker_cols = sorted(set(int(c) for r, c in pts2 if (r == 0 if marker_side == "top" else r == h - 1)))
        sign = 1 if marker_side == "top" else -1
        for c in range(w):
            crossed = sum(1 for mc in marker_cols if (mc <= c if anchor_side == "left" else mc >= c))
            shift = sign * crossed
            for br in base_rows:
                r = br + shift
                if 0 <= r < h and arr[r, c] != 2:
                    out[r, c] = 8

    out[arr == 2] = 2
    return out.tolist()

for split in ["train", "test", "arc-gen"]:
    ok = sum(python_rule(ex["input"]) == ex["output"] for ex in task[split])
    print(split, ok, "/", len(task[split]))


train 3 / 3
test 1 / 1
arc-gen 262 / 262


In [8]:

class Task382Model(nn.Module):
    def __init__(self, h=H, w=W, max_shift=9):
        super().__init__()
        self.max_shift = max_shift
        rr = torch.arange(h, dtype=torch.float32).view(1, 1, h, 1).expand(1, 1, h, w)
        cc = torch.arange(w, dtype=torch.float32).view(1, 1, 1, w).expand(1, 1, h, w)
        self.register_buffer("R", rr)
        self.register_buffer("C", cc)
        self.register_buffer("top_static", (rr == 0.0).float())
        self.register_buffer("left_static", (cc == 0.0).float())

        # Static shift matrices avoid Roll/Slice shape ambiguity. For rows:
        # out[r, c] = in[r-dr, c]. For cols: out[r, c] = in[r, c-dc].
        for s in range(-max_shift, max_shift + 1):
            row = torch.zeros(h, h, dtype=torch.float32)
            for dest in range(h):
                src = dest - s
                if 0 <= src < h:
                    row[dest, src] = 1.0
            col = torch.zeros(w, w, dtype=torch.float32)
            for src in range(w):
                dest = src + s
                if 0 <= dest < w:
                    col[src, dest] = 1.0
            self.register_buffer(f"row_shift_{s+max_shift}", row.view(1, 1, h, h))
            self.register_buffer(f"col_shift_{s+max_shift}", col.view(1, 1, w, w))

        row_down = torch.zeros(h, h, dtype=torch.float32)  # count markers k <= r
        row_up = torch.zeros(h, h, dtype=torch.float32)    # count markers k >= r
        for r in range(h):
            for k in range(h):
                if k <= r:
                    row_down[r, k] = 1.0
                if k >= r:
                    row_up[r, k] = 1.0
        col_right = torch.zeros(w, w, dtype=torch.float32) # count markers k <= c
        col_left = torch.zeros(w, w, dtype=torch.float32)  # count markers k >= c
        for k in range(w):
            for c in range(w):
                if k <= c:
                    col_right[k, c] = 1.0
                if k >= c:
                    col_left[k, c] = 1.0
        self.register_buffer("row_cumsum_down", row_down.view(1, 1, h, h))
        self.register_buffer("row_cumsum_up", row_up.view(1, 1, h, h))
        self.register_buffer("col_cumsum_right", col_right.view(1, 1, w, w))
        self.register_buffer("col_cumsum_left", col_left.view(1, 1, w, w))

    def _shift_rows(self, t, dr):
        mat = getattr(self, f"row_shift_{dr+self.max_shift}")
        return torch.matmul(mat, t)

    def _shift_cols(self, t, dc):
        mat = getattr(self, f"col_shift_{dc+self.max_shift}")
        return torch.matmul(t, mat)

    def _row_cumsum_down(self, t):
        return torch.matmul(self.row_cumsum_down, t)

    def _row_cumsum_up(self, t):
        return torch.matmul(self.row_cumsum_up, t)

    def _col_cumsum_right(self, t):
        return torch.matmul(t, self.col_cumsum_right)

    def _col_cumsum_left(self, t):
        return torch.matmul(t, self.col_cumsum_left)

    def _shifted_by_count_cols(self, base, count, direction):
        # base: [B,1,1,W] anchor columns expanded to each row.
        expanded = base + torch.zeros_like(count)
        y = torch.zeros_like(count)
        for s in range(self.max_shift + 1):
            eq = ((count - float(s)).abs() < 0.25).float()
            y = y + eq * self._shift_cols(expanded, direction * s)
        return (y > 0.5).float()

    def _shifted_by_count_rows(self, base, count, direction):
        # base: [B,1,H,1] anchor rows expanded to each column.
        expanded = base + torch.zeros_like(count)
        y = torch.zeros_like(count)
        for s in range(self.max_shift + 1):
            eq = ((count - float(s)).abs() < 0.25).float()
            y = y + eq * self._shift_rows(expanded, direction * s)
        return (y > 0.5).float()

    def forward(self, x):
        active = (x.sum(dim=1, keepdim=True) > 0.5).float()
        active_below = self._shift_rows(active, -1)
        active_right = self._shift_cols(active, -1)
        top = active * self.top_static
        left = active * self.left_static
        bottom = active * (1.0 - active_below)
        right = active * (1.0 - active_right)

        x8 = x[:, 8:9, :, :] * active
        x2 = x[:, 2:3, :, :] * active

        has_top8 = ((x8 * top).sum(dim=(2, 3), keepdim=True) > 0.5).float()
        has_bottom8 = ((x8 * bottom).sum(dim=(2, 3), keepdim=True) > 0.5).float()
        has_left8 = ((x8 * left).sum(dim=(2, 3), keepdim=True) > 0.5).float()
        has_right8 = ((x8 * right).sum(dim=(2, 3), keepdim=True) > 0.5).float()
        has_top2 = ((x2 * top).sum(dim=(2, 3), keepdim=True) > 0.5).float()
        has_bottom2 = ((x2 * bottom).sum(dim=(2, 3), keepdim=True) > 0.5).float()
        has_left2 = ((x2 * left).sum(dim=(2, 3), keepdim=True) > 0.5).float()
        has_right2 = ((x2 * right).sum(dim=(2, 3), keepdim=True) > 0.5).float()

        top_base_cols = ((x8 * top).sum(dim=2, keepdim=True) > 0.5).float()
        bottom_base_cols = ((x8 * bottom).sum(dim=2, keepdim=True) > 0.5).float()
        left_base_rows = ((x8 * left).sum(dim=3, keepdim=True) > 0.5).float()
        right_base_rows = ((x8 * right).sum(dim=3, keepdim=True) > 0.5).float()

        left_marker_rows = ((x2 * left).sum(dim=3, keepdim=True) > 0.5).float()
        right_marker_rows = ((x2 * right).sum(dim=3, keepdim=True) > 0.5).float()
        top_marker_cols = ((x2 * top).sum(dim=2, keepdim=True) > 0.5).float()
        bottom_marker_cols = ((x2 * bottom).sum(dim=2, keepdim=True) > 0.5).float()

        # Counts of crossed markers along the propagation direction, computed
        # with fixed triangular matrices rather than dynamic loops.
        left_rows_down = self._row_cumsum_down(left_marker_rows)
        right_rows_down = self._row_cumsum_down(right_marker_rows)
        left_rows_up = self._row_cumsum_up(left_marker_rows)
        right_rows_up = self._row_cumsum_up(right_marker_rows)

        top_cols_right = self._col_cumsum_right(top_marker_cols)
        bottom_cols_right = self._col_cumsum_right(bottom_marker_cols)
        top_cols_left = self._col_cumsum_left(top_marker_cols)
        bottom_cols_left = self._col_cumsum_left(bottom_marker_cols)

        y_top_left = self._shifted_by_count_cols(top_base_cols, left_rows_down + torch.zeros_like(active), +1)
        y_top_right = self._shifted_by_count_cols(top_base_cols, right_rows_down + torch.zeros_like(active), -1)
        y_bottom_left = self._shifted_by_count_cols(bottom_base_cols, left_rows_up + torch.zeros_like(active), +1)
        y_bottom_right = self._shifted_by_count_cols(bottom_base_cols, right_rows_up + torch.zeros_like(active), -1)

        y_left_top = self._shifted_by_count_rows(left_base_rows, top_cols_right + torch.zeros_like(active), +1)
        y_left_bottom = self._shifted_by_count_rows(left_base_rows, bottom_cols_right + torch.zeros_like(active), -1)
        y_right_top = self._shifted_by_count_rows(right_base_rows, top_cols_left + torch.zeros_like(active), +1)
        y_right_bottom = self._shifted_by_count_rows(right_base_rows, bottom_cols_left + torch.zeros_like(active), -1)

        y8 = (
            (has_top8 * has_left2) * y_top_left +
            (has_top8 * has_right2) * y_top_right +
            (has_bottom8 * has_left2) * y_bottom_left +
            (has_bottom8 * has_right2) * y_bottom_right +
            (has_left8 * has_top2) * y_left_top +
            (has_left8 * has_bottom2) * y_left_bottom +
            (has_right8 * has_top2) * y_right_top +
            (has_right8 * has_bottom2) * y_right_bottom
        )
        y8 = (y8 > 0.5).float() * active * (1.0 - x2)
        ch2 = x2
        ch0 = active * (1.0 - ch2) * (1.0 - y8)
        z = torch.zeros_like(ch0)
        return torch.cat([ch0, z, ch2, z, z, z, z, z, y8, z], dim=1)

model = Task382Model().eval()


In [9]:
dummy = torch.from_numpy(grid_to_tensor_zero_padded(task["test"][0]["input"]))

torch.onnx.export(
    model,
    dummy,
    str(ONNX_PATH),
    input_names=["input"],
    output_names=["output"],
    opset_version=17,
    do_constant_folding=True,
    dynamic_axes=None,
    dynamo=False,
)

# Save shape-inferred model so intermediate value_info tensors are statically described.
onnx_model = onnx.load(str(ONNX_PATH))
onnx_model = onnx.shape_inference.infer_shapes(onnx_model)
onnx.save(onnx_model, str(ONNX_PATH))
onnx.checker.check_model(str(ONNX_PATH))

ONNX_PATH, ONNX_PATH.stat().st_size


/tmp/ipykernel_16/2362384443.py:3: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


(PosixPath('/kaggle/working/task382_triangular_wave_stripe_active_canvas_30x30/task382.onnx'),
 282449)

In [10]:
def vi_shape(vi):
    dims = []
    for d in vi.type.tensor_type.shape.dim:
        if d.dim_value:
            dims.append(int(d.dim_value))
        elif d.dim_param:
            dims.append(str(d.dim_param))
        else:
            dims.append(None)
    return dims

onnx_model = onnx.load(str(ONNX_PATH))
ops = collections.Counter(node.op_type for node in onnx_model.graph.node)
forbidden = {"Loop", "Scan", "NonZero", "Unique", "Script", "Function"}
empty_inputs = [
    (node.name, node.op_type, list(node.input))
    for node in onnx_model.graph.node
    if any(inp == "" for inp in node.input)
]
bad_shapes = []
for vi in list(onnx_model.graph.input) + list(onnx_model.graph.value_info) + list(onnx_model.graph.output):
    shp = vi_shape(vi)
    if any(d is None or isinstance(d, str) for d in shp):
        bad_shapes.append((vi.name, shp))

print("input shape:", vi_shape(onnx_model.graph.input[0]))
print("output shape:", vi_shape(onnx_model.graph.output[0]))
print("ONNX size:", ONNX_PATH.stat().st_size)
print("ops:", dict(ops))
print("forbidden ops:", sorted(forbidden & set(ops)))
print("empty optional inputs:", len(empty_inputs))
print("non-static tensor shapes:", len(bad_shapes))

assert vi_shape(onnx_model.graph.input[0]) == [1, 10, 30, 30]
assert vi_shape(onnx_model.graph.output[0]) == [1, 10, 30, 30]
assert not (forbidden & set(ops))
assert not empty_inputs
assert not bad_shapes
assert ONNX_PATH.stat().st_size < 1_440_000


input shape: [1, 10, 30, 30]
output shape: [1, 10, 30, 30]
ONNX size: 282449
ops: {'Identity': 21, 'Constant': 227, 'ReduceSum': 17, 'Greater': 26, 'Cast': 106, 'MatMul': 90, 'Mul': 114, 'Sub': 84, 'Slice': 2, 'Add': 103, 'Abs': 80, 'Less': 80, 'Concat': 1}
forbidden ops: []
empty optional inputs: 0
non-static tensor shapes: 0


In [11]:

sess_options = ort.SessionOptions()
sess_options.intra_op_num_threads = 1
sess_options.inter_op_num_threads = 1
sess = ort.InferenceSession(str(ONNX_PATH), sess_options=sess_options, providers=["CPUExecutionProvider"])

def validate_examples(examples):
    tensor_ok = 0
    grid_ok = 0
    outside_input_active_zero_ok = 0
    outside_expected_canvas_zero_ok = 0
    bad = []
    for i, ex in enumerate(examples):
        x = grid_to_tensor_zero_padded(ex["input"])
        y = sess.run(None, {"input": x})[0]
        exp = grid_to_tensor_zero_padded(ex["output"])
        pred_bin = (y > 0.5).astype(np.float32)

        if np.array_equal(pred_bin, exp):
            tensor_ok += 1
        else:
            bad.append(i)

        h, w = len(ex["output"]), len(ex["output"][0])
        pred_grid = pred_bin[0, :, :h, :w].argmax(axis=0).astype(np.int64).tolist()
        if pred_grid == ex["output"]:
            grid_ok += 1

        input_active = x.sum(axis=1, keepdims=True) > 0.5
        expected_active = exp.sum(axis=1, keepdims=True) > 0.5
        if np.all(np.abs(y * (~input_active)) < 1e-5):
            outside_input_active_zero_ok += 1
        if np.all(np.abs(y * (~expected_active)) < 1e-5):
            outside_expected_canvas_zero_ok += 1

    return {
        "tensor_exact_zero_padded": [tensor_ok, len(examples)],
        "grid_argmax_inside_output_canvas": [grid_ok, len(examples)],
        "outside_input_active_all_channels_zero": [outside_input_active_zero_ok, len(examples)],
        "outside_expected_output_canvas_all_channels_zero": [outside_expected_canvas_zero_ok, len(examples)],
        "bad_indices": bad[:10],
    }

def validate_split(split):
    return validate_examples(task[split])

rng = random.Random(0)
arcgen_indices = list(range(len(task["arc-gen"])))
rng.shuffle(arcgen_indices)
holdout_n = max(1, int(math.ceil(0.60 * len(arcgen_indices))))
arcgen_holdout = [task["arc-gen"][i] for i in arcgen_indices[:holdout_n]]

summary = {
    "task_id": TASK_ID,
    "rule": RULE_DESCRIPTION,
    "onnx_path": str(ONNX_PATH),
    "onnx_size_bytes": ONNX_PATH.stat().st_size,
    "input_shape": vi_shape(onnx_model.graph.input[0]),
    "output_shape": vi_shape(onnx_model.graph.output[0]),
    "ops": dict(ops),
    "forbidden_ops": sorted(forbidden & set(ops)),
    "empty_optional_inputs": len(empty_inputs),
    "non_static_tensor_shapes": len(bad_shapes),
    "arc_gen_holdout_policy": "deterministic random seed 0, 60% of arc-gen; full arc-gen also validated",
    "validation": {
        "train": validate_split("train"),
        "test": validate_split("test"),
        "arc-gen_60pct_holdout": validate_examples(arcgen_holdout),
        "arc-gen_full": validate_split("arc-gen"),
    },
}

print(json.dumps(summary, indent=2)[:6000])
with SUMMARY_PATH.open("w") as f:
    json.dump(summary, f, indent=2)

for split_name, result in summary["validation"].items():
    assert result["tensor_exact_zero_padded"][0] == result["tensor_exact_zero_padded"][1], split_name
    assert result["outside_input_active_all_channels_zero"][0] == result["outside_input_active_all_channels_zero"][1], split_name
    assert result["outside_expected_output_canvas_all_channels_zero"][0] == result["outside_expected_output_canvas_all_channels_zero"][1], split_name


{
  "task_id": "task382",
  "rule": "bouncing diagonal / triangular-wave stripe propagation + active-canvas masking",
  "onnx_path": "/kaggle/working/task382_triangular_wave_stripe_active_canvas_30x30/task382.onnx",
  "onnx_size_bytes": 282449,
  "input_shape": [
    1,
    10,
    30,
    30
  ],
  "output_shape": [
    1,
    10,
    30,
    30
  ],
  "ops": {
    "Identity": 21,
    "Constant": 227,
    "ReduceSum": 17,
    "Greater": 26,
    "Cast": 106,
    "MatMul": 90,
    "Mul": 114,
    "Sub": 84,
    "Slice": 2,
    "Add": 103,
    "Abs": 80,
    "Less": 80,
    "Concat": 1
  },
  "forbidden_ops": [],
  "empty_optional_inputs": 0,
  "non_static_tensor_shapes": 0,
  "arc_gen_holdout_policy": "deterministic random seed 0, 60% of arc-gen; full arc-gen also validated",
  "validation": {
    "train": {
      "tensor_exact_zero_padded": [
        3,
        3
      ],
      "grid_argmax_inside_output_canvas": [
        3,
        3
      ],
      "outside_input_active_all_channels_

In [12]:
with zipfile.ZipFile(SUBMISSION_PATH, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(ONNX_PATH, arcname=f"{TASK_ID}.onnx")

print("Wrote:", SUBMISSION_PATH)
print("Zip contents:", zipfile.ZipFile(SUBMISSION_PATH).namelist())
assert zipfile.ZipFile(SUBMISSION_PATH).namelist() == [f"{TASK_ID}.onnx"]


Wrote: /kaggle/working/submission.zip
Zip contents: ['task382.onnx']
